In [0]:
count_df = df_transformed.groupBy("Type").count()
display(count_df)

Type,count
Pollution,4557031
Temperature,8842927
Traffic,6600042


In [0]:
df_transformed.write.mode("overwrite").saveAsTable("iot_cleaned")

In [0]:
anomaly_df = df_transformed.filter(df_transformed.Value > 150)
display(anomaly_df)

SensorID,Type,Value,Location,Timestamp
3,Traffic,184,East,2026-04-01
5,Traffic,198,East,2026-04-01
20,Pollution,162,North,2026-04-01
26,Traffic,197,South,2026-04-01
29,Traffic,163,South,2026-04-01
37,Traffic,168,North,2026-04-01
38,Pollution,196,East,2026-04-01
42,Traffic,175,North,2026-04-01
45,Temperature,185,East,2026-04-01
51,Temperature,156,East,2026-04-01


In [0]:
peak_df = df_transformed.groupBy("Location") \
    .avg("Value") \
    .orderBy("avg(Value)", ascending=False)

display(peak_df)

Location,avg(Value)
West,99.51874612085275
North,99.4971700373076
East,99.46010105830233
South,99.45657845252734


In [0]:
avg_df = df_transformed.groupBy("Type").avg("Value")
display(avg_df)

Type,avg(Value)
Pollution,99.45137502904852
Temperature,99.47730745713496
Traffic,99.48380873939892


In [0]:
from pyspark.sql.functions import col

df_transformed = df_clean.withColumn("Value", col("Value").cast("int"))

In [0]:
df_clean = df.dropna()

In [0]:
df = spark.table("iot_data")
df.show(5)


+--------+-----------+-----+--------+----------+
|SensorID|       Type|Value|Location| Timestamp|
+--------+-----------+-----+--------+----------+
|15000000|Temperature|   85|   North|2026-04-01|
|15000001|Temperature|  195|    East|2026-04-01|
|15000002|    Traffic|  121|    West|2026-04-01|
|15000003|Temperature|  133|    East|2026-04-01|
|15000004|    Traffic|   25|   South|2026-04-01|
+--------+-----------+-----+--------+----------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import rand, when, col, lit

rows = 20_000_000

df = spark.range(0, rows)

df = df.withColumn("SensorID", col("id")) \
    .withColumn("Type",
        when(rand() < 0.33, "Traffic")
        .when(rand() < 0.66, "Temperature")
        .otherwise("Pollution")) \
    .withColumn("Value", (rand()*200).cast("int")) \
    .withColumn("Location",
        when(rand() < 0.25, "North")
        .when(rand() < 0.5, "South")
        .when(rand() < 0.75, "East")
        .otherwise("West")) \
    .withColumn("Timestamp", lit("2026-04-01")) \
    .drop("id")

# ✅ SAVE AS TABLE (IMPORTANT)
df.write.mode("overwrite").saveAsTable("iot_data")

In [0]:
from pyspark.sql.functions import rand, when, col

rows = 20_000_000   # ~0.5–1GB depending on compression

df = spark.range(0, rows)

df = df.withColumn("SensorID", col("id")) \
    .withColumn("Type",
        when(rand() < 0.33, "Traffic")
        .when(rand() < 0.66, "Temperature")
        .otherwise("Pollution")) \
    .withColumn("Value", (rand()*200).cast("int")) \
    .withColumn("Location",
        when(rand() < 0.25, "North")
        .when(rand() < 0.5, "South")
        .when(rand() < 0.75, "East")
        .otherwise("West")) \
    .withColumn("Timestamp", "2026-04-01") \
    .drop("id")

df.write.mode("overwrite").csv("/FileStore/iot_data")

---------------------------------------------------------------------------
PySparkTypeError                          Traceback (most recent call last)
File <command-6149336635476282>, line 18
      3 rows = 20_000_000   # ~0.5–1GB depending on compression
      5 df = spark.range(0, rows)
      7 df = df.withColumn("SensorID", col("id")) \
      8     .withColumn("Type",
      9         when(rand() < 0.33, "Traffic")
     10         .when(rand() < 0.66, "Temperature")
     11         .otherwise("Pollution")) \
     12     .withColumn("Value", (rand()*200).cast("int")) \
     13     .withColumn("Location",
     14         when(rand() < 0.25, "North")
     15         .when(rand() < 0.5, "South")
     16         .when(rand() < 0.75, "East")
     17         .otherwise("West")) \
---> 18     .withColumn("Timestamp", "2026-04-01") \
     19     .drop("id")
     21 df.write.mode("overwrite").csv("/FileStore/iot_data")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/

In [0]:
df = spark.range(10)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+

